# Sesión 01 en vivo — LLMs y Setup de Entorno de Trabajo

**Programa AI Engineering — LIDR**

Este notebook contiene todos los ejercicios de la sesión en vivo. Ejecuta las celdas de configuración primero y luego ve avanzando bloque por bloque con el instructor.

---

## Configuración inicial

Ejecuta estas tres celdas antes de empezar. Necesitas tener tus API keys configuradas en **Colab Secrets** (icono de llave 🔑 en el panel lateral):
- `OPENAI_API_KEY` → tu key de OpenAI (empieza por `sk-`)
- `ANTHROPIC_API_KEY` → tu key de Anthropic (empieza por `sk-ant-`)

In [ ]:
# Install dependencies
# Dependency warnings from Colab pre-installed packages (gradio, typer) are normal and do not affect our code
!pip install -q openai anthropic tiktoken litellm 2>&1 | grep -v 'dependency' | grep -v 'WARNING' | tail -1

# Verify installation
import openai, anthropic, tiktoken, litellm
from importlib.metadata import version
print(f"✅ openai {openai.__version__}")
print(f"✅ anthropic {anthropic.__version__}")
print(f"✅ tiktoken {version('tiktoken')}")
print(f"✅ litellm {version('litellm')}")

In [ ]:
# Load API keys from Colab Secrets
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# Create clients
from openai import OpenAI
from anthropic import Anthropic

openai_client = OpenAI()
anthropic_client = Anthropic()

print("✅ Clients configured successfully")

In [ ]:
# Utility function to calculate costs (used throughout the session)

PRICING = {
    # OpenAI (USD per 1M tokens)
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4o-mini-2024-07-18": {"input": 0.15, "output": 0.60},
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
    "gpt-5.4": {"input": 2.50, "output": 15.00},
    "gpt-5.4-nano": {"input": 0.20, "output": 1.25},
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "gpt-5-nano": {"input": 0.05, "output": 0.40},
    # Anthropic (USD per 1M tokens)
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
    "claude-sonnet-4-6-20250514": {"input": 3.00, "output": 15.00},
}


def calculate_cost(model, input_tokens, output_tokens):
    """Calculate the cost of an API call in USD."""
    prices = PRICING.get(model)
    if not prices:
        for key in PRICING:
            if key in model or model in key:
                prices = PRICING[key]
                break
    if not prices:
        return None

    cost_in = (input_tokens / 1_000_000) * prices["input"]
    cost_out = (output_tokens / 1_000_000) * prices["output"]
    return {"input": cost_in, "output": cost_out, "total": cost_in + cost_out}


print("✅ Cost calculation function loaded")

---
# Bloque 2 — Del prompt del usuario al prompt del producto

**Objetivo:** La calidad de la respuesta no puede depender de que el usuario sepa hacer prompting. El desarrollador es quien diseña la experiencia.

### 2.1. El problema: la respuesta genérica

Enviamos una pregunta realista pero sin system prompt. ¿Pondríais esta respuesta en un producto?

In [ ]:
# No system prompt — the model has no context about what we need
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="We're migrating our payment service from a monolith to microservices. What should we consider?"
)

print(response.output_text)
print(f"\n--- {response.usage.input_tokens} input tokens + {response.usage.output_tokens} output tokens ---")

### 2.2. El system prompt como contrato de calidad

Vamos a enviar la **misma pregunta** con system prompts cada vez más elaborados. Cada iteración añade una capa de control sobre la respuesta.

In [ ]:
# The question stays the same across all iterations
USER_QUESTION = "We're migrating our payment service from a monolith to microservices. What should we consider?"

In [ ]:
# Iteration 1 — Basic role: slightly better, but still unfocused
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a senior software architect with experience in distributed systems.",
    input=USER_QUESTION
)

print("=== ITERATION 1: Basic role ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

In [ ]:
# Iteration 2 — Role + audience + format constraints
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="""You are a senior software architect with 15+ years of experience in distributed systems and payment infrastructure.

Your audience is a team of experienced backend developers (5+ years) who understand software fundamentals but have never done a monolith-to-microservices migration.

Response format:
- Write in prose, no bullet points or numbered lists
- Maximum 250 words
- Prioritize actionable insights over generic theory""",
    input=USER_QUESTION
)

print("=== ITERATION 2: Role + audience + format ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

In [ ]:
# Iteration 3 — Adding behavioral constraints and edge case handling
response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="""You are a senior software architect with 15+ years of experience in distributed systems and payment infrastructure. You have led multiple monolith-to-microservices migrations at fintech companies.

Your audience is a team of experienced backend developers (5+ years) who understand software fundamentals but have never done a monolith-to-microservices migration.

Response rules:
- Write in prose, no bullet points or numbered lists
- Maximum 250 words
- Prioritize actionable insights over generic theory
- When discussing trade-offs, always present both sides with concrete consequences
- If the question lacks critical context (team size, traffic volume, regulatory requirements), explicitly state what assumptions you are making
- Never recommend a specific vendor or product — focus on architectural patterns
- Always mention at least one non-obvious risk that teams commonly overlook""",
    input=USER_QUESTION
)

print("=== ITERATION 3: Role + audience + format + constraints ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

In [ ]:
# Iteration 4 — Production-grade prompt: full quality contract
PRODUCTION_SYSTEM_PROMPT = """You are a principal software architect specializing in payment systems, PCI-DSS compliance, and large-scale distributed architectures. You have led monolith-to-microservices migrations at three fintech companies processing over $1B in annual transactions.

Audience: senior backend developers (5+ years experience) planning their first major architectural migration. They understand distributed systems theory but need practical, battle-tested guidance.

Response structure:
- Open with the single most critical decision they need to make first
- Follow with 2-3 key considerations, each grounded in a concrete scenario or real-world consequence
- Close with one commonly overlooked risk and how to mitigate it
- Write in prose, no bullet points, no numbered lists, no markdown headers
- Maximum 300 words

Behavioral constraints:
- When you lack context (team size, current traffic, regulatory environment), state your assumptions explicitly before advising
- Never recommend specific vendors, cloud providers, or commercial products
- Present trade-offs with concrete consequences: 'If you choose X, expect Y; if you choose Z, expect W'
- Distinguish between what is essential on day one versus what can be deferred
- If the migration scope is ambiguous, ask one clarifying question before providing your analysis

Do not:
- Use filler phrases like 'Great question!' or 'There are many things to consider'
- Repeat information the user already provided
- Give generic advice that applies to any software project, not specifically to payment system migrations"""

response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=PRODUCTION_SYSTEM_PROMPT,
    input=USER_QUESTION
)

print("=== ITERATION 4: Production-grade prompt ===")
print(response.output_text)
print(f"\n[{response.usage.output_tokens} output tokens]")

### 2.3. Abstraer al usuario del prompting

En un producto real, el usuario no escribe prompts. Solo aporta datos mínimos. El desarrollador construye el prompt internamente. Veamos dos versiones de la misma funcionalidad: una amateur y una profesional.

In [ ]:
# NAIVE version — minimal prompt, inconsistent results

def review_pull_request_naive(diff_text: str) -> str:
    """Naive approach: vague instructions, unpredictable output."""
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions="You are a code reviewer.",
        input=f"Review this code:\n\n{diff_text}",
        temperature=0.7
    )
    return response.output_text


sample_diff = """def process_payment(amount, currency, user_id):
    if amount > 0:
        result = gateway.charge(amount, currency)
        db.save(user_id, result)
        return result
    return None"""

print("=== NAIVE VERSION ===")
print(review_pull_request_naive(sample_diff))

In [ ]:
# PRODUCTION version — detailed prompt, consistent and actionable output

def review_pull_request(diff_text: str, context: str = "") -> str:
    """
    Production-grade PR review.
    The user pastes a diff. The developer controls everything else.
    """
    instructions = """You are a staff engineer performing a pull request review for a fintech company that processes credit card payments. Your reviews are known for being thorough but respectful.

Review criteria (in priority order):
1. Security vulnerabilities: injection, authentication gaps, sensitive data exposure
2. Error handling: missing try/catch, silent failures, unhandled edge cases
3. Data integrity: race conditions, transaction boundaries, idempotency
4. Code clarity: naming, function length, single responsibility

Response format:
- Start with a one-sentence overall assessment (approve, request changes, or needs discussion)
- For each finding: describe the issue, explain the concrete risk, suggest a fix with a brief code snippet if applicable
- Maximum 3 findings — prioritize by severity, skip nitpicks
- Write in prose, no markdown headers, no bullet points

Constraints:
- Never say 'looks good' if there are security or data integrity concerns
- If the diff is too small to identify the full context, state what assumptions you are making
- Do not suggest style changes unless they affect readability in a meaningful way
- Treat the author as a competent developer — explain the 'why', not the 'how to write code'"""

    user_input = f"Pull request diff to review:\n\n```\n{diff_text}\n```"
    if context:
        user_input += f"\n\nAdditional context: {context}"

    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=instructions,
        input=user_input,
        temperature=0.15,
        max_output_tokens=600
    )
    return response.output_text


# The user only pastes a diff — the developer controls everything else
print("=== PRODUCTION VERSION ===")
print(review_pull_request(sample_diff, context="This function runs inside our payment processing pipeline."))

print("\n" + "=" * 60)
print("The user pasted 6 lines of code.")
print("The model received a 250-word system prompt + formatted input + context.")

### 🧪 Tu turno

Crea tu propia función que abstraiga el prompting con un prompt de nivel producción. Algunas ideas:
- Analizar un requisito de software y detectar ambigüedades
- Evaluar un diseño de base de datos y sugerir mejoras
- Revisar un mensaje de error de cara al usuario y proponer uno mejor
- Analizar un endpoint API y evaluar su diseño REST

In [ ]:
# Your production-grade function here

def my_function(user_input: str) -> str:
    instructions = """..."""

    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=instructions,
        input=user_input,
        temperature=0.2,
        max_output_tokens=500
    )
    return response.output_text

# Test:
# print(my_function("..."))

### 2.4. Consistencia de respuestas

Los modelos son estocásticos. ¿Es aceptable que dos usuarios reciban respuestas de calidad diferente ante la misma pregunta? Veamos cómo la combinación de temperature + instrucciones precisas controla la variabilidad.

In [ ]:
# High temperature + vague prompt = wildly inconsistent responses
print("=== HIGH TEMPERATURE (1.0) + VAGUE PROMPT ===")
for i in range(4):
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions="You are a technical consultant.",
        input="Should we use microservices?",
        temperature=1.0
    )
    # Show first 120 chars to compare at a glance
    text = response.output_text.replace("\n", " ")[:120]
    print(f"  Run {i+1}: {text}...")

In [ ]:
# Low temperature + precise prompt = consistent, predictable responses
print("=== LOW TEMPERATURE (0.1) + PRECISE PROMPT ===")
for i in range(4):
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions="""You are a solutions architect. When asked about architecture decisions, always respond with exactly three sentences:
1. State the key trade-off in one sentence
2. Give the scenario where option A wins
3. Give the scenario where option B wins
Do not use bullet points. Write the three sentences as a single paragraph.""",
        input="Should we use microservices?",
        temperature=0.1
    )
    text = response.output_text.replace("\n", " ")[:120]
    print(f"  Run {i+1}: {text}...")

---
# Bloque 3 — Parámetros del modelo y herramientas integradas

**Objetivo:** Conocer los parámetros esenciales que controlan el comportamiento del modelo y las herramientas integradas que la API pone a nuestra disposición.

Parámetros que veremos:
- `temperature` — controla la aleatoriedad (con nota sobre `top_p` como alternativa)
- `max_output_tokens` — límite máximo de tokens de salida
- `stop` y `seed` — control de parada y reproducibilidad

Herramientas integradas que veremos:
- `web_search` — acceso a información actual de la web
- Mención breve de otras: `file_search`, `code_interpreter`, `computer_use`

### 3.1. Temperature: de determinista a creativo

`temperature` controla cuánta aleatoriedad introduce el modelo al elegir el siguiente token. El rango típico es 0.0 – 2.0 (en Anthropic, 0.0 – 1.0).

- **0.0** — el modelo siempre elige el token más probable. Respuestas casi deterministas.
- **0.3 – 0.5** — ligera variación. Bueno para análisis técnico, clasificación, extracción de datos.
- **0.7 – 1.0** — respuestas más variadas y creativas. Bueno para brainstorming, redacción creativa.
- **> 1.0** — alta aleatoriedad. Puede producir resultados incoherentes.

**Alternativa: `top_p` (nucleus sampling).** Limita el muestreo a los tokens cuya probabilidad acumulada supere `p`. Con `top_p=0.1`, solo se consideran los tokens más probables. **Regla:** usa `temperature` O `top_p`, nunca ambos a la vez — tanto OpenAI como Anthropic lo recomiendan explícitamente, y en modelos Claude 4.5+ su uso combinado es directamente rechazado por la API.

Veamos el efecto con una pregunta idéntica lanzada 3 veces a cada nivel:

In [ ]:
TEMP_SYSTEM = "You are a product naming consultant. Propose a single product name for the given description. Respond with only the name, no explanation."
TEMP_QUESTION = "A mobile app that helps freelance developers track billable hours across multiple clients."

temperatures = [0.0, 0.7, 1.5]

for temp in temperatures:
    print(f"\n=== temperature = {temp} ===")
    for i in range(3):
        resp = openai_client.responses.create(
            model="gpt-4o-mini",
            instructions=TEMP_SYSTEM,
            input=TEMP_QUESTION,
            temperature=temp,
            max_output_tokens=30
        )
        print(f"  Run {i+1}: {resp.output_text.strip()}")

**Observaciones esperadas:**
- Con `temperature=0.0`, las 3 respuestas son idénticas o casi idénticas.
- Con `temperature=0.7`, las respuestas son diferentes pero todas razonables.
- Con `temperature=1.5`, las respuestas pueden ser muy creativas — o incoherentes.

**Regla práctica:** empieza siempre con `temperature=0.2-0.3` para tareas de producción. Sube solo cuando necesites creatividad explícita.

### 3.2. max_output_tokens: controlando la longitud de la respuesta

`max_output_tokens` define el **techo** de tokens que el modelo puede generar. No es la longitud objetivo — es el límite superior. Si se alcanza, la respuesta se corta abruptamente.

Dos razones para usarlo en producción:
1. **Control de coste** — los tokens de salida son 3-10x más caros que los de entrada.
2. **Prevención de respuestas runaway** — evita que el modelo genere respuestas excesivamente largas.

In [ ]:
LENGTH_SYSTEM = "You are a technical writer. Explain the concept clearly and thoroughly."
LENGTH_QUESTION = "Explain how database indexing works and when to use different index types."

limits = [50, 200, 800]

for limit in limits:
    resp = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=LENGTH_SYSTEM,
        input=LENGTH_QUESTION,
        temperature=0.3,
        max_output_tokens=limit
    )

    status = resp.status
    truncated = " [TRUNCATED]" if status == "incomplete" else " [COMPLETE]"

    print(f"\n=== max_output_tokens = {limit}{truncated} ===")
    print(f"Tokens generated: {resp.usage.output_tokens}")
    print(f"Response:\n{resp.output_text}")

**Observaciones esperadas:**
- Con `max_output_tokens=50`, la respuesta se corta a mitad de frase. El `status` será `"incomplete"`.
- Con `max_output_tokens=800`, la respuesta termina de forma natural antes de llegar al límite.

**Regla práctica:** `max_output_tokens` es un límite de seguridad, no un control de longitud. Para controlar la longitud real, combínalo con instrucciones en el system prompt ("Maximum 150 words", "Respond in exactly 3 sentences").

⚠️ **Importante:** en Anthropic, `max_tokens` es obligatorio. En OpenAI y Gemini es opcional.

### 3.3. Otros parámetros útiles: stop y seed

**`stop` (OpenAI) / `stop_sequences` (Anthropic):** lista de strings. Si el modelo genera cualquiera, la generación se detiene inmediatamente. El string de parada **no** se incluye en la respuesta. Útil para garantizar un formato específico (por ejemplo, parar al encontrar `"###"` o `"END"`) o para ahorrar tokens cortando en un delimitador conocido.

**`seed`:** parámetro de OpenAI y Gemini (**no existe en Anthropic**). Misma entrada + mismo seed = misma respuesta. Imprescindible para tests automatizados que involucran LLMs — sin él, tus tests serán flaky. OpenAI no garantiza determinismo al 100% entre snapshots, pero en la práctica es muy consistente.

In [ ]:
# Example 1: stop — truncate a list before the 3rd item
print("=== Without stop ===")
resp = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a technical assistant.",
    input="List 5 advantages of microservices architecture. Number each one.",
    temperature=0.3,
    max_output_tokens=300
)
print(resp.output_text)
print(f"\n[Output tokens: {resp.usage.output_tokens}]")

print("\n=== With stop=['3.'] — cuts before item 3 ===")
resp = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a technical assistant.",
    input="List 5 advantages of microservices architecture. Number each one.",
    temperature=0.3,
    max_output_tokens=300,
    stop=["3."]
)
print(resp.output_text)
print(f"\n[Output tokens: {resp.usage.output_tokens}]")

In [ ]:
# Example 2: seed — reproducible outputs even with high temperature
SEED_SYSTEM = "You are a creative writer."
SEED_QUESTION = "Write a one-sentence product tagline for a time-tracking app."

print("=== Same seed (42) — reproducible output ===")
for i in range(3):
    resp = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=SEED_SYSTEM,
        input=SEED_QUESTION,
        temperature=0.8,
        seed=42,
        max_output_tokens=50
    )
    print(f"  Run {i+1}: {resp.output_text.strip()}")

print("\n=== Different seeds — varied output ===")
for seed in [1, 100, 9999]:
    resp = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=SEED_SYSTEM,
        input=SEED_QUESTION,
        temperature=0.8,
        seed=seed,
        max_output_tokens=50
    )
    print(f"  seed={seed}: {resp.output_text.strip()}")

**Regla práctica:**
- **`stop`** es útil cuando quieres garantizar un formato específico sin depender de que el modelo "decida" cuándo parar.
- **`seed`** debería estar en todos tus tests automatizados que involucren LLMs.

### 3.4. Resumen: reglas prácticas

Configuraciones típicas por tipo de tarea:

| Tarea | `temperature` | `max_output_tokens` | `seed` |
|-------|:-------------:|:-------------------:|:------:|
| Análisis / clasificación | 0.0 – 0.2 | 200 – 500 | Sí (para tests) |
| Extracción de datos | 0.0 | 500 – 1500 | Sí |
| Resúmenes técnicos | 0.2 – 0.4 | 500 – 1000 | Opcional |
| Chat conversacional | 0.3 – 0.5 | 500 – 1500 | No |
| Redacción creativa | 0.7 – 0.9 | 1000 – 3000 | No |

No hay valores "correctos" universales — son puntos de partida que se ajustan midiendo calidad real en tu caso de uso.

⚠️ **Nota sobre modelos de razonamiento:** los modelos de razonamiento (OpenAI o-series, GPT-5 series, Claude con extended thinking) **no aceptan** la mayoría de estos parámetros. Temperature, top_p, frequency_penalty y otros están fijados internamente. En su lugar, ofrecen `reasoning_effort` (OpenAI) o `thinking.budget_tokens` (Anthropic). Tenemos una guía dedicada a este tema en el material asíncrono.

### 3.5. Herramientas integradas del modelo

Hasta ahora, nuestros LLMs solo responden con lo que saben del entrenamiento. Pero los LLMs tienen tres limitaciones fundamentales:

1. **Knowledge cutoff** — no saben nada posterior a su fecha de entrenamiento
2. **No acceden a información privada** — no pueden leer tus documentos, base de datos, etc.
3. **No ejecutan código** — no pueden calcular cosas, ejecutar SQL, procesar archivos

Las **herramientas integradas** resuelven estas limitaciones. Son funcionalidades que el proveedor ejecuta en sus propios servidores: el modelo decide cuándo usarlas, las invoca, procesa los resultados y genera la respuesta final. Todo en una sola llamada a la API desde tu lado.

**Herramientas disponibles (abril 2026):**

| Herramienta | OpenAI | Anthropic | Para qué sirve |
|-------------|:------:|:---------:|----------------|
| `web_search` | ✅ | ✅ | Buscar información actual en internet |
| `file_search` / retrieval | ✅ (con vector store) | ❌ (usa MCP) | Buscar en tus documentos |
| `code_interpreter` / `code_execution` | ✅ | ✅ | Ejecutar código Python en sandbox |
| `computer_use` | ✅ | ✅ (beta) | Controlar un ordenador (clicks, teclado) |
| `image_generation` | ✅ | ❌ | Generar imágenes |
| MCP remote servers | ✅ | ✅ | Conectar a servidores externos de herramientas |

Vamos a ver la más inmediatamente útil: `web_search`.

### 3.6. web_search con OpenAI: respuestas con información actual

La pregunta "¿qué pasó ayer en la bolsa?" es imposible de responder con el conocimiento del modelo. Con `web_search`, el modelo busca, lee páginas web relevantes, y genera una respuesta con citas.

Sintaxis en OpenAI: añades `tools=[{"type": "web_search"}]` a la llamada. El modelo decide si usar la herramienta o no.

In [ ]:
# Question that requires current information
response = openai_client.responses.create(
    model="gpt-4o",
    input="What were the most important announcements at OpenAI DevDay 2025?",
    tools=[{"type": "web_search"}]
)

print("=== Response ===")
print(response.output_text)
print(f"\n=== Tokens ===")
print(f"Input: {response.usage.input_tokens} / Output: {response.usage.output_tokens}")

**Citas de fuentes:** la respuesta incluye automáticamente `annotations` que referencian las URLs consultadas. Puedes acceder a ellas iterando sobre `response.output`:

In [ ]:
# Extract the URLs the model consulted
print("=== Sources consulted ===")
for item in response.output:
    if item.type == "message":
        for block in item.content:
            if hasattr(block, "annotations") and block.annotations:
                for annotation in block.annotations:
                    if hasattr(annotation, "url"):
                        print(f"  - {annotation.url}")

**Observaciones importantes:**
- La API gestiona todo el flujo: decide si buscar, formula la query, lee resultados, sintetiza
- El modelo cita sus fuentes automáticamente (fundamental para productos serios)
- No todos los modelos soportan `web_search`. En el ejemplo usamos `gpt-4o`. Si pruebas con `gpt-4o-mini` puede fallar.
- Tiene coste adicional sobre los tokens normales (consulta precios actualizados en la documentación)

### 3.7. web_search con Anthropic: mismo patrón, distinta sintaxis

Anthropic ofrece la misma funcionalidad con una sintaxis ligeramente distinta. La diferencia más importante: la herramienta tiene que estar **habilitada por el administrador de la organización** en la Claude Console antes de poder usarse.

In [ ]:
# Note: requires web search enabled in your organization's Claude Console
# If you get an error, the feature is not enabled on your account yet.

try:
    response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": "What were the most important announcements at OpenAI DevDay 2025?"
        }],
        tools=[{
            "type": "web_search_20250305",
            "name": "web_search",
            "max_uses": 3   # Cap on number of searches
        }]
    )

    # The response may contain multiple content blocks (text + tool_use + text)
    for block in response.content:
        if block.type == "text":
            print(block.text)

    # Check how many searches were performed
    if hasattr(response.usage, "server_tool_use"):
        print(f"\n[Web searches performed: {response.usage.server_tool_use.web_search_requests}]")
    print(f"[Tokens: {response.usage.input_tokens} in / {response.usage.output_tokens} out]")

except Exception as e:
    print(f"Error: {e}")
    print("\nIf the error says web_search is not enabled, ask your org admin to enable it")
    print("in the Claude Console (Settings > Privacy > Tools).")

**Diferencias prácticas entre ambas implementaciones:**

| Aspecto | OpenAI | Anthropic |
|---------|--------|-----------|
| Sintaxis del tool | `{"type": "web_search"}` | `{"type": "web_search_20250305", "name": "web_search"}` |
| Versionado | Implícito en el SDK | Explícito en el tool type (fecha) |
| Habilitación | Directa con API key | Requiere enable en Console |
| Control de uso | Automático | `max_uses` explícito |
| Dominios permitidos | Configurable | `allowed_domains` / `blocked_domains` |
| Pricing | Incluido en el pricing del modelo | $10 / 1.000 búsquedas + tokens |
| Citas | `annotations` en output | Citas inline en el texto |

### 3.8. Otras herramientas disponibles

No las ejecutaremos en clase, pero es importante conocerlas porque aparecerán en sesiones posteriores del programa:

**`file_search` (OpenAI) — búsqueda en tus documentos**

```python
# Requires creating a Vector Store first with your documents
response = openai_client.responses.create(
    model="gpt-4o",
    input="What is our policy on refund requests?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": ["<your-vector-store-id>"]
    }]
)
```

Es RAG gestionado por OpenAI: subes documentos, OpenAI los chunkea, vectoriza e indexa. El modelo busca fragmentos relevantes automáticamente. **Conectaremos con esto en el Módulo 3 cuando construyamos nuestro propio sistema RAG.**

**`code_interpreter` / `code_execution` — ejecutar código en un sandbox**

```python
# OpenAI
tools=[{"type": "code_interpreter", "container": {"type": "auto"}}]

# Anthropic
tools=[{"type": "code_execution_20250825", "name": "code_execution"}]
```

El modelo genera código Python, lo ejecuta en un entorno aislado, usa el resultado para responder. Útil para cálculos precisos, análisis de CSVs, procesamiento de archivos. **Resuelve el problema del modelo que no sabe sumar.**

**`computer_use` — controlar un ordenador**

El modelo puede ver screenshots, mover el ratón, escribir, hacer clicks. Sirve para automatizar tareas en interfaces gráficas. Todavía en beta en ambos proveedores.

**MCP (Model Context Protocol) remote servers**

Ambos proveedores soportan conectar el modelo a servidores MCP remotos. Es el estándar emergente para exponer herramientas personalizadas al modelo. **Lo veremos en el Módulo 5 cuando trabajemos con agentes.**

---

**Regla general:** las herramientas integradas transforman al LLM de "oráculo que sabe" a "agente que actúa". Son el puente entre un modelo estático y un producto realmente útil.

---
# Bloque 4 — Comparativa de proveedores en vivo

**Objetivo:** Experimentar las diferencias reales entre proveedores y entre tiers de modelos con la misma pregunta y el mismo prompt profesional.

### 4.1. Misma pregunta, distinto proveedor

Usamos un prompt de producción y una pregunta compleja para ver diferencias reales, no respuestas de juguete.

In [ ]:
# Shared prompt and question for provider comparison
COMPARISON_SYSTEM_PROMPT = """You are a database architect with deep experience in both relational and NoSQL systems in production environments handling 10K+ requests per second.

When comparing technologies:
- Ground every claim in a concrete scenario (e.g., 'if your write volume exceeds X, then Y becomes a bottleneck because Z')
- Quantify trade-offs when possible (latency ranges, storage overhead percentages, operational cost implications)
- Distinguish between limitations that are fundamental vs. limitations that tooling can mitigate
- Write in prose, no bullet points, no headers
- Maximum 200 words
- Do not hedge with 'it depends' without specifying what it depends on"""

COMPARISON_QUESTION = """We have a SaaS platform with 50M rows of customer activity data growing at 2M rows/month. 
Currently on PostgreSQL 16, queries are getting slow on our analytics dashboard. 
Should we migrate the analytics workload to a columnar store or add read replicas and optimize indexes?"""

In [ ]:
# OpenAI (Responses API)
openai_response = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=COMPARISON_SYSTEM_PROMPT,
    input=COMPARISON_QUESTION,
    temperature=0.2,
    max_output_tokens=400
)

openai_cost = calculate_cost(
    openai_response.model,
    openai_response.usage.input_tokens,
    openai_response.usage.output_tokens
)

print("=== OpenAI (gpt-4o-mini) ===")
print(openai_response.output_text)
print(f"\nModel: {openai_response.model}")
print(f"Tokens: {openai_response.usage.input_tokens} input + {openai_response.usage.output_tokens} output")
if openai_cost:
    print(f"Cost: ${openai_cost['total']:.6f}")

In [ ]:
# Anthropic (Messages API)
anthropic_response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=COMPARISON_SYSTEM_PROMPT,
    messages=[{"role": "user", "content": COMPARISON_QUESTION}],
    max_tokens=400,
    temperature=0.2
)

anthropic_cost = calculate_cost(
    anthropic_response.model,
    anthropic_response.usage.input_tokens,
    anthropic_response.usage.output_tokens
)

print("=== Anthropic (claude-haiku-4-5) ===")
print(anthropic_response.content[0].text)
print(f"\nModel: {anthropic_response.model}")
print(f"Tokens: {anthropic_response.usage.input_tokens} input + {anthropic_response.usage.output_tokens} output")
if anthropic_cost:
    print(f"Cost: ${anthropic_cost['total']:.6f}")

In [ ]:
# Side-by-side comparison
print("=" * 70)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 70)
print(f"{'':30s} {'OpenAI':>15s}  {'Anthropic':>15s}")
print("-" * 70)
print(f"{'Input tokens':30s} {openai_response.usage.input_tokens:>15d}  {anthropic_response.usage.input_tokens:>15d}")
print(f"{'Output tokens':30s} {openai_response.usage.output_tokens:>15d}  {anthropic_response.usage.output_tokens:>15d}")
if openai_cost and anthropic_cost:
    print(f"{'Total cost':30s} {'$'+f'{openai_cost["total"]:.6f}':>15s}  {'$'+f'{anthropic_cost["total"]:.6f}':>15s}")

### 4.2. Misma pregunta, distinto tier de modelo

Comparamos modelos de distintos precios dentro de OpenAI con la misma pregunta compleja. ¿Justifica la diferencia de precio la diferencia de calidad?

In [ ]:
# Compare different OpenAI model tiers with a complex question
openai_models = [
    "gpt-4o-mini",
    # "gpt-5.4-mini",   # Uncomment if you have access
    # "gpt-5.4",        # Uncomment if you have access (more expensive)
]

for model_name in openai_models:
    try:
        resp = openai_client.responses.create(
            model=model_name,
            instructions=COMPARISON_SYSTEM_PROMPT,
            input=COMPARISON_QUESTION,
            temperature=0.2,
            max_output_tokens=400
        )

        cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)

        print(f"\n{'=' * 60}")
        print(f"Model: {resp.model}")
        print(f"Tokens: {resp.usage.input_tokens} input + {resp.usage.output_tokens} output")
        if cost:
            print(f"Cost: ${cost['total']:.6f}")
        print(f"\nResponse:\n{resp.output_text}")
    except Exception as e:
        print(f"\n{'=' * 60}")
        print(f"Model {model_name}: Error — {e}")

### 🧪 Tu turno

Haz la misma comparación con modelos de Anthropic. Prueba con `claude-haiku-4-5-20251001` y si tienes acceso, con `claude-sonnet-4-6-20250514`.

In [ ]:
# Compare different Anthropic model tiers
anthropic_models = [
    "claude-haiku-4-5-20251001",
    # "claude-sonnet-4-6-20250514",  # Uncomment if you have access
]

for model_name in anthropic_models:
    try:
        resp = anthropic_client.messages.create(
            model=model_name,
            system=COMPARISON_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": COMPARISON_QUESTION}],
            max_tokens=400,
            temperature=0.2
        )

        cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)

        print(f"\n{'=' * 60}")
        print(f"Model: {resp.model}")
        print(f"Tokens: {resp.usage.input_tokens} input + {resp.usage.output_tokens} output")
        if cost:
            print(f"Cost: ${cost['total']:.6f}")
        print(f"\nResponse:\n{resp.content[0].text}")
    except Exception as e:
        print(f"\n{'=' * 60}")
        print(f"Model {model_name}: Error — {e}")

---
# Bloque 5 — Agregadores: una interfaz, múltiples proveedores

**Objetivo:** Cambiar de proveedor y modelo con una sola línea de código usando LiteLLM.

In [ ]:
from litellm import completion

AGGREGATOR_SYSTEM_PROMPT = """You are a backend engineering lead evaluating infrastructure decisions.
Respond in exactly 3 sentences: the recommendation, the primary reason, and one risk to watch for.
Be specific — reference concrete thresholds, metrics, or failure modes. No generic advice."""

AGGREGATOR_QUESTION = "We process 500 webhook events per second from Stripe. Should we handle them synchronously in our Rails API or push them to a message queue first?"

# Same function, same interface — only the model string changes
providers = [
    "openai/gpt-4o-mini",
    "anthropic/claude-haiku-4-5-20251001",
]

for model_name in providers:
    try:
        resp = completion(
            model=model_name,
            messages=[
                {"role": "system", "content": AGGREGATOR_SYSTEM_PROMPT},
                {"role": "user", "content": AGGREGATOR_QUESTION}
            ],
            temperature=0.2,
            max_tokens=250
        )

        print(f"\n{'=' * 60}")
        print(f"Model: {model_name}")
        print(f"Tokens: {resp.usage.prompt_tokens} input + {resp.usage.completion_tokens} output")
        print(f"\nResponse:\n{resp.choices[0].message.content}")
    except Exception as e:
        print(f"\n{'=' * 60}")
        print(f"{model_name}: Error — {e}")

### Observación importante

LiteLLM usa la interfaz de **Chat Completions** como formato unificado:
- `messages` en lugar de `instructions` + `input`
- `resp.choices[0].message.content` en lugar de `resp.output_text`
- `resp.usage.prompt_tokens` en lugar de `resp.usage.input_tokens`

Por eso es importante conocer ambas interfaces de OpenAI: la **Responses API** es la que usamos directamente, y **Chat Completions** es el "lenguaje común" de los agregadores.

### Wrapper con cambio de proveedor en una línea

Así se ve una función de producción que abstrae completamente el proveedor:

In [ ]:
def evaluate_technical_decision(
    scenario: str,
    model: str = "openai/gpt-4o-mini"
) -> str:
    """
    Evaluate a technical decision given a scenario description.
    Switching providers = changing the model string. No other code changes.
    """
    resp = completion(
        model=model,
        messages=[
            {"role": "system", "content": """You are a principal engineer performing a technical design review.
For every decision, provide: the recommended approach, one concrete reason grounded in production experience, and one thing that could go wrong.
Maximum 3 sentences. No bullet points."""},
            {"role": "user", "content": scenario}
        ],
        temperature=0.2,
        max_tokens=200
    )
    return resp.choices[0].message.content


scenario = "We want to store user session data. Considering Redis vs PostgreSQL for a system with 20K concurrent users."

# Switching providers = changing one string
print("OpenAI:")
print(evaluate_technical_decision(scenario, "openai/gpt-4o-mini"))
print()
print("Anthropic:")
print(evaluate_technical_decision(scenario, "anthropic/claude-haiku-4-5-20251001"))

---
# Bloque 6 — Tokens: cómo funcionan y cuánto cuestan

**Objetivo:** Entender los tokens de forma práctica — qué son, cómo se cuentan, y cómo impactan en coste.

### 6.1. Anatomía de los tokens

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")

sample_texts = {
    "Short English": "How much does it cost to migrate a database?",
    "Short Spanish": "¿Cuánto cuesta migrar una base de datos?",
    "Python code": "def migrate_db(source, target):\n    return source.copy_to(target)",
    "JSON payload": '{"project": "migration", "estimate_days": 45, "confidence": "medium"}',
    "Production prompt": PRODUCTION_SYSTEM_PROMPT,  # The prompt we built in Block 2
}

print(f"{'Text':25s} {'Characters':>12s} {'Tokens':>8s} {'Tok/word':>10s}")
print("-" * 59)

for label, text in sample_texts.items():
    tokens = enc.encode(text)
    word_count = len(text.split())
    ratio = len(tokens) / word_count if word_count > 0 else 0
    print(f"{label:25s} {len(text):>12d} {len(tokens):>8d} {ratio:>9.1f}x")

In [ ]:
# Visualize how a text is decomposed into tokens
sample = "¿Cuánto cuesta migrar una base de datos PostgreSQL a Amazon Aurora?"
tokens = enc.encode(sample)

print(f"Text: {sample}")
print(f"Total: {len(tokens)} tokens\n")
print("Decomposition:")
for i, t in enumerate(tokens):
    print(f"  Token {i}: '{enc.decode([t])}' (id: {t})")

### 🧪 Tu turno

Tokeniza textos en tu idioma, con jerga técnica de tu dominio, o el system prompt de la función que creaste en el bloque 2. ¿Cuántos tokens consume?

In [ ]:
# Your text here
my_text = "..."

tokens = enc.encode(my_text)
print(f"'{my_text}' -> {len(tokens)} tokens")

### 6.2. Input vs output: la asimetría que define tu factura

In [ ]:
# Experiment: same question, different output limits
TOKEN_SYSTEM_PROMPT = "You are a technical assistant. Respond in Spanish."
TOKEN_QUESTION = "Explain the CAP theorem and when each trade-off matters."

output_limits = [50, 200, 1000]

print(f"{'max_output_tokens':>20s} {'Input tok':>12s} {'Output tok':>12s} {'Cost':>12s}")
print("-" * 60)

for limit in output_limits:
    resp = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=TOKEN_SYSTEM_PROMPT,
        input=TOKEN_QUESTION,
        max_output_tokens=limit,
        temperature=0.3
    )

    cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)
    cost_str = f"${cost['total']:.6f}" if cost else "N/A"

    print(f"{limit:>20d} {resp.usage.input_tokens:>12d} {resp.usage.output_tokens:>12d} {cost_str:>12s}")

In [ ]:
# Scale projection: how much does this cost in production?

USERS = 10_000
CALLS_PER_USER_PER_DAY = 5
CALLS_PER_DAY = USERS * CALLS_PER_USER_PER_DAY
DAYS_PER_MONTH = 30
CALLS_PER_MONTH = CALLS_PER_DAY * DAYS_PER_MONTH

# Average call: 150 input tokens (production prompt is longer), 200 output tokens
AVG_INPUT_TOKENS = 150
AVG_OUTPUT_TOKENS = 200

models_to_compare = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-5.4-mini": {"input": 0.75, "output": 4.50},
    "gpt-5.4": {"input": 2.50, "output": 15.00},
    "claude-haiku-4-5": {"input": 1.00, "output": 5.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
}

print(f"Scenario: {USERS:,} users x {CALLS_PER_USER_PER_DAY} calls/day x {DAYS_PER_MONTH} days")
print(f"Total: {CALLS_PER_MONTH:,} calls/month")
print(f"Average per call: {AVG_INPUT_TOKENS} input tokens + {AVG_OUTPUT_TOKENS} output tokens")
print(f"\n{'Model':25s} {'Cost/call':>15s} {'Cost/month':>15s}")
print("-" * 58)

for model_name, prices in models_to_compare.items():
    cost_per_call = (
        (AVG_INPUT_TOKENS / 1_000_000) * prices["input"] +
        (AVG_OUTPUT_TOKENS / 1_000_000) * prices["output"]
    )
    monthly_cost = cost_per_call * CALLS_PER_MONTH
    print(f"{model_name:25s} ${cost_per_call:>14.6f} ${monthly_cost:>13.2f}")

### 6.3. El coste oculto de las conversaciones multi-turno

In [ ]:
# Multi-turn conversation: watch input tokens grow with each turn

conversation_history = []
conversation_system_prompt = """You are a DevOps engineer helping a team debug a production incident.
Ask clarifying questions when needed. Keep each response under 3 sentences.
Focus on actionable next steps, not theory."""

questions = [
    "Our API response times spiked from 200ms to 3s about 30 minutes ago.",
    "PostgreSQL CPU is at 95%. No recent deployments.",
    "We found a query doing a sequential scan on a 40M row table. It was fine yesterday.",
    "The table was auto-vacuumed last night. Could that have invalidated the index statistics?",
    "Running ANALYZE on the table now. What else should we check while we wait?",
]

print(f"{'Turn':>6s} {'Input tok':>12s} {'Output tok':>12s} {'Cumulative cost':>18s}")
print("-" * 52)

cumulative_cost = 0

for i, question in enumerate(questions):
    conversation_history.append({"role": "user", "content": question})

    resp = openai_client.responses.create(
        model="gpt-4o-mini",
        instructions=conversation_system_prompt,
        input=conversation_history,
        max_output_tokens=200,
        temperature=0.2
    )

    conversation_history.append({"role": "assistant", "content": resp.output_text})

    cost = calculate_cost(resp.model, resp.usage.input_tokens, resp.usage.output_tokens)
    if cost:
        cumulative_cost += cost["total"]

    print(f"{i+1:>6d} {resp.usage.input_tokens:>12d} {resp.usage.output_tokens:>12d} ${cumulative_cost:>16.6f}")

In [ ]:
# Visualize the cost of the system prompt across turns
system_prompt_tokens = len(enc.encode(conversation_system_prompt))

print(f"\n📈 Input token growth per turn:")
print(f"Each turn resends the entire conversation history as input tokens.")
print(f"\nSystem prompt in this example: ~{system_prompt_tokens} tokens")
print(f"The production prompt from Block 2: ~{len(enc.encode(PRODUCTION_SYSTEM_PROMPT))} tokens")
print(f"\nOver a 20-turn conversation with the production prompt:")
print(f"  System prompt alone = {len(enc.encode(PRODUCTION_SYSTEM_PROMPT))} x 20 = {len(enc.encode(PRODUCTION_SYSTEM_PROMPT)) * 20:,} input tokens")

---
# Bloque 7 — Análisis de arquitecturas reales (discusión)

Este bloque es de discusión guiada por el instructor — no hay código que ejecutar.

**Preguntas para analizar cada producto:**

1. ¿Cómo abstrae el prompting del usuario?
2. ¿Qué modelo probablemente usa?
3. ¿Cómo gestiona los tokens?
4. ¿Usa model routing (modelos diferentes para tareas diferentes)?

**Productos sugeridos:**
- GitHub Copilot
- Asistente de soporte (Intercom / Zendesk AI)
- Herramienta de análisis de documentos (NotebookLM)

---
# Extra: Resumen de las tres interfaces API

Referencia rápida de las tres interfaces que hemos usado hoy.

In [ ]:
# The three API interfaces used today, side by side:

SUMMARY_SYSTEM = """You are a senior engineer. When asked a technical question, respond with exactly one sentence that a principal engineer would find useful. No filler."""
SUMMARY_QUESTION = "What is the most common mistake teams make when adopting Kubernetes?"

# 1. OpenAI Responses API (recommended for OpenAI direct usage)
resp1 = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=SUMMARY_SYSTEM,        # system prompt as top-level parameter
    input=SUMMARY_QUESTION,              # user input as string
    max_output_tokens=100,
    temperature=0.1
)
print("1. Responses API:", resp1.output_text)
print(f"   Tokens: {resp1.usage.input_tokens} in / {resp1.usage.output_tokens} out")
print(f"   Status: {resp1.status}")

print()

# 2. Anthropic Messages API
resp2 = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=SUMMARY_SYSTEM,               # system prompt as separate parameter
    messages=[{"role": "user", "content": SUMMARY_QUESTION}],  # messages array
    max_tokens=100,                      # MANDATORY in Anthropic
    temperature=0.1
)
print("2. Messages API:", resp2.content[0].text)
print(f"   Tokens: {resp2.usage.input_tokens} in / {resp2.usage.output_tokens} out")
print(f"   Stop reason: {resp2.stop_reason}")

print()

# 3. LiteLLM (unified Chat Completions interface)
resp3 = completion(
    model="openai/gpt-4o-mini",
    messages=[
        {"role": "system", "content": SUMMARY_SYSTEM},   # system as message in array
        {"role": "user", "content": SUMMARY_QUESTION}
    ],
    max_tokens=100,
    temperature=0.1
)
print("3. LiteLLM:", resp3.choices[0].message.content)
print(f"   Tokens: {resp3.usage.prompt_tokens} in / {resp3.usage.completion_tokens} out")
print(f"   Finish reason: {resp3.choices[0].finish_reason}")